# Phase 3: Autograd

Small examples for the completed reverse-mode autodiff features.

In [1]:
from babygrad.nn.activations import ReLU, Softmax
from babygrad.nn.losses import CCE
from babygrad.nn.modules import Linear, Sequential
from babygrad.tensor import Tensor
from babygrad.types import NodeKind

## The computational graph

Every operation is an `Op` node: it holds the input tensors, the op label, and the `backward()` gradient rule. The tensor it creates points back at it through `producer` — that link is the whole graph.

In [2]:
a = Tensor([1, 2, 3, 4], shape=(2, 2), kind=NodeKind.PARAMETER)
b = Tensor([5, 6, 7, 8], shape=(2, 2), kind=NodeKind.PARAMETER)
c = a * b

{
    "op": c.producer.label,
    "inputs": c.producer.inputs,
}

{'op': '*',
 'inputs': [2 rows x 2 cols
  [
    [1  2]
    [3  4]
  ],
  2 rows x 2 cols
  [
    [5  6]
    [7  8]
  ]]}

Leaf tensors created by hand have no producer — they are the roots the graph walk stops at.

In [3]:
a.producer is None

True

## Backward pass

`backward()` is called once on the final tensor. It seeds that tensor's gradient with ones, then walks the graph in reverse, accumulating each parent's `.grad`.

In [4]:
x = Tensor([1, 2, 3, 4], shape=(2, 2), kind=NodeKind.PARAMETER)
y = Tensor([10, 20, 30, 40], shape=(2, 2), kind=NodeKind.PARAMETER)

loss = (x * y).sum()
loss.backward()

{
    "x.grad": Tensor(x.grad, shape=x.shape, kind=NodeKind.VIEW),
    "y.grad": Tensor(y.grad, shape=y.shape, kind=NodeKind.VIEW),
}

{'x.grad': 2 rows x 2 cols
 [
   [10.0  20.0]
   [30.0  40.0]
 ],
 'y.grad': 2 rows x 2 cols
 [
   [1.0  2.0]
   [3.0  4.0]
 ]}

## Gradient accumulation

A tensor used by multiple consumers receives a gradient contribution from each path. Propagation to its own parents only happens after every child has delivered, so contributions accumulate with `+=` instead of overwriting.

In [5]:
a = Tensor([2.0, 3.0], shape=(2,), kind=NodeKind.PARAMETER)

# 'a' feeds both the multiply (twice) and the add: d(a*a + a)/da = 2a + 1
out = (a * a + a).sum()
out.backward()

a.grad

[5.0, 7.0]

## Reduction gradients

Reductions spread the incoming gradient back across the group they collapsed. `mean` divides it evenly; `max` routes it only to the winning elements, splitting ties.

In [6]:
t = Tensor([1, 2, 3, 4, 5, 6], shape=(2, 3), kind=NodeKind.PARAMETER)
t.mean(axis=1).backward()

Tensor(t.grad, shape=t.shape, kind=NodeKind.VIEW)

2 rows x 3 cols
[
  [0.3333  0.3333  0.3333]
  [0.3333  0.3333  0.3333]
]

In [7]:
t = Tensor([1, 5, 5, 7, 2, 0], shape=(2, 3), kind=NodeKind.PARAMETER)
t.max(axis=1).backward()

# row 0 has tied winners, so they split the gradient
Tensor(t.grad, shape=t.shape, kind=NodeKind.VIEW)

2 rows x 3 cols
[
  [0.0  0.5  0.5]
  [1.0  0.0  0.0]
]

## Matmul and transpose

Matmul gradients are themselves matmuls against the transposed counterpart, so shapes always line up with the parents.

In [8]:
A = Tensor([1, 2, 3, 4, 5, 6], shape=(2, 3), kind=NodeKind.PARAMETER)
B = Tensor([1, 0, 0, 1, 1, 1], shape=(3, 2), kind=NodeKind.PARAMETER)

C = A @ B
C.backward()

{
    "A.grad": Tensor(A.grad, shape=A.shape, kind=NodeKind.VIEW),
    "B.grad": Tensor(B.grad, shape=B.shape, kind=NodeKind.VIEW),
}

{'A.grad': 2 rows x 3 cols
 [
   [1.0  1.0  2.0]
   [1.0  1.0  2.0]
 ],
 'B.grad': 3 rows x 2 cols
 [
   [5.0  5.0]
   [7.0  7.0]
   [9.0  9.0]
 ]}

In [9]:
x = Tensor([1, 2, 3, 4, 5, 6], shape=(2, 3), kind=NodeKind.PARAMETER)

# transpose participates in the graph: d((x.T)**2)/dx = 2x, delivered back in x's layout
(x.t() ** 2).sum().backward()

Tensor(x.grad, shape=x.shape, kind=NodeKind.VIEW)

2 rows x 3 cols
[
  [2.0   4.0   6.0]
  [8.0  10.0  12.0]
]

## ReLU and softmax

ReLU is an op node like any other; the `nn.ReLU` layer is a thin wrapper around it. Its gradient rule passes the gradient through where the input was positive and blocks it elsewhere.

In [10]:
x = Tensor([-2, -1, 0, 3, 4, -5], shape=(2, 3), kind=NodeKind.PARAMETER)

ReLU().forward(x).sum().backward()

Tensor(x.grad, shape=x.shape, kind=NodeKind.VIEW)

2 rows x 3 cols
[
  [0.0  0.0  0.0]
  [1.0  1.0  0.0]
]

Softmax needs no gradient rule of its own — it is composed entirely from primitive ops (`max`, `sub`, `exp`, `sum`, `div`), so its output already carries a producer chain.

In [11]:
logits = Tensor([0, 0, 0, 2, 2, 2], shape=(2, 3), kind=NodeKind.PARAMETER)
probabilities = Softmax().forward(logits)

probabilities.producer.label

'/'

## End to end: model gradients from a loss

Everything composes: a forward pass through a small model, a loss, and one `backward()` call fills in the gradients for every weight and bias.

In [12]:
hidden = Linear(input_size=2, output_size=4)
output = Linear(input_size=4, output_size=3)
model = Sequential([hidden, ReLU(), output, Softmax()])

inputs = Tensor([1, 2, 3, 4], shape=(2, 2), kind=NodeKind.INPUT)
targets = Tensor([0, 1, 0, 1, 0, 0], shape=(2, 3), kind=NodeKind.TARGET)

y_pred = model.forward(inputs)
loss = CCE().forward(targets, y_pred)
loss.backward()

{
    "loss": loss,
    "hidden.weights.grad": Tensor(hidden.weights.grad, shape=hidden.weights.shape, kind=NodeKind.VIEW),
    "hidden.bias.grad": Tensor(hidden.bias.grad, shape=hidden.bias.shape, kind=NodeKind.VIEW),
    "output.weights.grad": Tensor(output.weights.grad, shape=output.weights.shape, kind=NodeKind.VIEW),
    "output.bias.grad": Tensor(output.bias.grad, shape=output.bias.shape, kind=NodeKind.VIEW),
}

{'loss': 1 rows x 1 cols
 [
   [1.6448]
 ],
 'hidden.weights.grad': 2 rows x 4 cols
 [
   [0.4631  0.0  0.6085  0.0]
   [0.9513  0.0  1.2495  0.0]
 ],
 'hidden.bias.grad': 1 rows x 4 cols
 [
   [0.4883  0.0  0.6411  0.0]
 ],
 'output.weights.grad': 4 rows x 3 cols
 [
   [0.4571  -0.5846  0.1275]
   [   0.0      0.0     0.0]
   [  0.43  -0.5351  0.1051]
   [   0.0      0.0     0.0]
 ],
 'output.bias.grad': 1 rows x 3 cols
 [
   [0.3949  -0.4792  0.0843]
 ]}